In [2]:
import cv2
import homcloud.interface as hc
import tqdm
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
import scipy
from scipy import stats
from scipy.spatial import distance_matrix
import imageio
import codecs
import copy
import math
import napari
import networkx as nx
from skimage.segmentation import watershed
import tqdm

In [ ]:
### eroding the 3D module output
from skimage.morphology import erosion, ball

def eroded_marker(label_img, N=3):
    markers = np.zeros_like(label_img, dtype='uint8')
    labels = np.unique(label_img)
    labels = labels[labels != 0]

    for lab in labels:
        mask = (label_img == lab)
        eroded = mask.copy()
        for _ in range(N):
            eroded = erosion(eroded, ball(1)) 
            if not eroded.any():
                break
        markers[eroded] = lab

    return markers


In [ ]:
### smoothing based on gaussian filter
import numpy as np
from scipy.ndimage import distance_transform_edt, gaussian_filter

def smooth_multiclass_label(label, sigma=1.0):
    classes = np.unique(label)

    dist_stack = []
    for c in classes:
        mask = (label == c)
        dist = distance_transform_edt(~mask) - distance_transform_edt(mask)
        dist_smooth = gaussian_filter(dist, sigma=sigma)
        dist_stack.append(dist_smooth)

    dist_stack = np.stack(dist_stack, axis=0)  # (n_class, Z, Y, X)
    smoothed_label = classes[np.argmin(dist_stack, axis=0)]

    return smoothed_label.astype(label.dtype)

In [ ]:
### TimeSeriesLabels_tracked: 3D module output labels after tracking process
### TimeSeriesImages: Original time series images used for watershed method
def smooth_label_generation(i):
    labels2=TimeSeriesLabels_tracked[i]
    eroded_markers=eroded_marker(labels2, N=5)
    eroded_markers[0,0,0]=9### This number should be larger than the largest label index. Here, we had 8 cells, thus we chose 9.
    new_labels = watershed(TimeSeriesImages[i], eroded_markers)
    new_labels[new_labels==9]=0
    smoothed_labels=smooth_multiclass_label(new_labels,sigma=5)
    return smoothed_labels

In [12]:
timeseries_smoothed_labels=[]
for NUMBER in tqdm.tqdm(range (40,74)):
    smoothed_labels=smooth_label_generation(NUMBER-40)
    timeseries_smoothed_labels.append(smoothed_labels)

100%|██████████| 34/34 [15:18<00:00, 27.02s/it]


In [13]:
timeseries_smoothed_labels=np.array(timeseries_smoothed_labels).astype('uint8')